# LLM Redis Worker (Jupyter DataLab)

Воркер LLM-моста redis-bridge: слушает stream заявок в Redis, исполняет их
против целевого LLM-бэкенда (GigaChat / OpenAI-совместимый) и возвращает
ответ. Протокол — `docs/integrations/redis-llm-bridge.md`.

**Запуск:** Run All. Остановка: прервать ячейку запуска (Kernel → Interrupt) —
heartbeat-ключ удалится, приложение мгновенно увидит «воркер недоступен».


In [ ]:
# Зависимости (раскомментировать при первом запуске, если пакетов нет)
# %pip install redis httpx


In [ ]:
"""Конфигурация. Всё берётся из env; можно переопределить константами."""
import os
import socket

# Цели: имя цели = проводной формат (openai | gigachat).
# Цель публикуется в heartbeat, если задан url; token опционален
# (локальный sglang/vLLM без авторизации — легальная цель).
TARGETS = {
    "gigachat": {
        "url": os.environ.get("GIGACHAT_API_URL", ""),
        "token": os.environ.get("JPY_API_TOKEN", ""),
        "rate_limit_sec": 5.0,   # GigaChat: не чаще 1 запроса в 5 сек
    },
    "openai": {
        "url": os.environ.get("OPENAI_API_URL", ""),
        "token": os.environ.get("OPENAI_API_KEY", ""),
        "rate_limit_sec": 0.0,   # sglang/vLLM: без лимита
    },
}

REDIS_HOST = os.environ.get("BRIDGE_REDIS_HOST", "10.110.10.38")
REDIS_PORT = int(os.environ.get("BRIDGE_REDIS_PORT", "7474"))
REDIS_PASSWORD = os.environ.get("BRIDGE_REDIS_PASSWORD", "") or None

KEY_PREFIX = "llm:bridge:"
CONSUMER_GROUP = "llm-workers"
WORKER_ID = f"{socket.gethostname()}:{os.getpid()}"

HEARTBEAT_INTERVAL_SEC = 15
HEARTBEAT_TTL_SEC = 45
HEALTH_CHECK_TIMEOUT_SEC = 4    # GET /models цели в heartbeat-такте
MAX_ATTEMPTS = 3            # попыток вызова LLM на заявку
RETRY_PAUSES_SEC = [5, 10, 20]
RESP_TTL_SEC = 300
HTTP_TIMEOUT_SEC = 120
# XAUTOCLAIM: подобрать зависшее. Worst-case живой обработки одной заявки
# ~6 мин (HTTP 120с × 3 попытки-обрыва + паузы 5/10/20с + rate-limit) —
# порог 10 мин гарантирует, что клеймится только заведомо брошенное,
# а не заявка, которую другой воркер ещё обрабатывает (иначе дубль-вызовы).
CLAIM_MIN_IDLE_MS = 600_000

def available_targets():
    return [n for n, t in TARGETS.items() if t["url"]]

print(f"Воркер {WORKER_ID}; доступные цели: {available_targets()}")


In [ ]:
"""Логика воркера: heartbeat + consumer."""
import asyncio
import json
import time

import httpx
import redis.asyncio as aioredis

stats = {
    "processed": 0,
    "errors": 0,
    "started_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
_last_call_at: dict[str, float] = {}


def make_redis() -> aioredis.Redis:
    # socket_timeout НЕ задаём: XREADGROUP BLOCK держит сокет дольше 5 сек.
    return aioredis.Redis(
        host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD,
        decode_responses=True,
    )


def auth_headers(cfg: dict) -> dict:
    """Authorization только при заданном token (sglang/vLLM живут без него)."""
    return {"Authorization": f"Bearer {cfg['token']}"} if cfg["token"] else {}


async def check_target_health(http: httpx.AsyncClient, name: str) -> bool:
    """GET /models цели: жив ли LLM-бэкенд за воркером.

    Здоровье = ответ со статусом < 500 (401/404 значит «сервер отвечает»).
    Rate limit НЕ трогаем: это дешёвый GET, не completion.
    """
    cfg = TARGETS[name]
    try:
        resp = await http.get(
            cfg["url"].rstrip("/") + "/models",
            headers=auth_headers(cfg),
            timeout=HEALTH_CHECK_TIMEOUT_SEC,
        )
        return resp.status_code < 500
    except httpx.HTTPError:
        return False


async def heartbeat_loop(r: aioredis.Redis) -> None:
    """Продлевает ключ-статус каждые 15 сек — параллельно любой обработке.

    Вместе со статусом публикует target_health: health probe приложения
    закрывает circuit breaker, только когда жив не только воркер,
    но и LLM-бэкенд за ним (иначе breaker «хлопал» бы).
    """
    async with httpx.AsyncClient() as http:
        while True:
            target_health = {
                name: await check_target_health(http, name)
                for name in available_targets()
            }
            payload = {
                "worker_id": WORKER_ID,
                "started_at": stats["started_at"],
                "last_beat": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "processed": stats["processed"],
                "errors": stats["errors"],
                "targets": available_targets(),
                "target_health": target_health,
            }
            await r.set(
                KEY_PREFIX + "worker:alive",
                json.dumps(payload, ensure_ascii=False),
                ex=HEARTBEAT_TTL_SEC,
            )
            await asyncio.sleep(HEARTBEAT_INTERVAL_SEC)


async def call_llm(http: httpx.AsyncClient, target: str, path: str,
                   body: dict, deadline_ts: float):
    """POST к цели с rate limit и ретраями. Возвращает ("final", resp) |
    ("error", status_code, message) | ("expired", None, None)."""
    cfg = TARGETS[target]
    last_error = (502, "нет попыток")
    for attempt in range(MAX_ATTEMPTS):
        wait = cfg["rate_limit_sec"] - (time.time() - _last_call_at.get(target, 0.0))
        if wait > 0:
            await asyncio.sleep(wait)
        if time.time() > deadline_ts:
            return ("expired", None, None)
        _last_call_at[target] = time.time()
        try:
            resp = await http.post(
                cfg["url"].rstrip("/") + path,
                json=body,
                headers=auth_headers(cfg),
            )
        except httpx.HTTPError as exc:
            last_error = (502, f"сетевая ошибка: {exc!r}")
        else:
            if resp.status_code < 400:
                return ("final", resp, None)
            if resp.status_code == 429 or resp.status_code >= 500:
                last_error = (resp.status_code, resp.text[:500])
            else:  # 4xx — не ретраим
                return ("error", resp.status_code, resp.text[:500])
        if attempt < MAX_ATTEMPTS - 1:
            pause = RETRY_PAUSES_SEC[min(attempt, len(RETRY_PAUSES_SEC) - 1)]
            if time.time() + pause > deadline_ts:
                break
            print(f"  повтор через {pause}с (попытка {attempt + 2})")
            await asyncio.sleep(pause)
    return ("error", last_error[0], last_error[1])


async def process_entry(r, http, entry_id: str, fields: dict) -> None:
    req_id = fields.get("id", "?")
    target = fields.get("target", "")
    resp_key = KEY_PREFIX + "resp:" + req_id
    received_ts = time.time()

    async def reply(payload: dict) -> None:
        await r.xadd(resp_key, {"v": "1", "seq": "0",
                                "received_ts": str(received_ts), **payload})
        await r.expire(resp_key, RESP_TTL_SEC)

    try:
        deadline_ts = float(fields.get("deadline_ts", "0") or 0)
        if deadline_ts and time.time() > deadline_ts:
            print(f"[{req_id}] просрочена, пропуск")
            return
        if target not in available_targets():
            stats["errors"] += 1
            await reply({"kind": "error", "status_code": "503",
                         "message": f"цель {target!r} не настроена",
                         "started_ts": str(time.time()),
                         "finished_ts": str(time.time())})
            return
        body = json.loads(fields["body"])
        started_ts = time.time()
        outcome, a, b = await call_llm(
            http, target, fields.get("path", "/chat/completions"),
            body, deadline_ts or (time.time() + 600),
        )
        finished_ts = time.time()
        if outcome == "expired":
            print(f"[{req_id}] дедлайн истёк во время обработки")
            return
        if outcome == "final":
            stats["processed"] += 1
            await reply({"kind": "final", "status_code": str(a.status_code),
                         "body": a.text,
                         "started_ts": str(started_ts),
                         "finished_ts": str(finished_ts)})
            print(f"[{req_id}] ok за {finished_ts - started_ts:.1f}с")
        else:
            stats["errors"] += 1
            await reply({"kind": "error", "status_code": str(a),
                         "message": str(b),
                         "started_ts": str(started_ts),
                         "finished_ts": str(finished_ts)})
            print(f"[{req_id}] ошибка {a}")
    except Exception as exc:  # заявка не должна ронять воркер
        stats["errors"] += 1
        try:
            await reply({"kind": "error", "status_code": "500",
                         "message": f"внутренняя ошибка воркера: {exc!r}",
                         "started_ts": str(time.time()),
                         "finished_ts": str(time.time())})
        except Exception:
            print(f"[{req_id}] не удалось записать ответ: {exc!r}")
    finally:
        await r.xack(KEY_PREFIX + "requests", CONSUMER_GROUP, entry_id)


async def consumer_loop(r: aioredis.Redis) -> None:
    stream = KEY_PREFIX + "requests"
    try:
        await r.xgroup_create(stream, CONSUMER_GROUP, id="$", mkstream=True)
    except aioredis.ResponseError as exc:
        if "BUSYGROUP" not in str(exc):
            raise
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT_SEC) as http:
        while True:
            # 1) подобрать зависшее (упавший воркер взял и не доделал)
            _next, claimed, _deleted = await r.xautoclaim(
                stream, CONSUMER_GROUP, WORKER_ID,
                min_idle_time=CLAIM_MIN_IDLE_MS, start_id="0-0", count=10,
            )
            for entry_id, fields in claimed:
                print(f"подобрана зависшая заявка {entry_id}")
                await process_entry(r, http, entry_id, fields)
            # 2) новые заявки
            batch = await r.xreadgroup(
                CONSUMER_GROUP, WORKER_ID, {stream: ">"}, count=1, block=5000,
            )
            for _stream_name, entries in batch or []:
                for entry_id, fields in entries:
                    await process_entry(r, http, entry_id, fields)


In [ ]:
"""Запуск. Остановка — Kernel → Interrupt: heartbeat-ключ удаляется."""
r = make_redis()
hb_task = asyncio.create_task(heartbeat_loop(r))
consumer_task = asyncio.create_task(consumer_loop(r))
try:
    await asyncio.gather(hb_task, consumer_task)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("Останавливаюсь...")
finally:
    # Сначала гасим ОБА таска, потом снимаем ключ: иначе при упавшем
    # consumer'е живой heartbeat заново создал бы worker:alive после DEL —
    # зомби-статус воркера, который ничего не потребляет.
    hb_task.cancel()
    consumer_task.cancel()
    await asyncio.gather(hb_task, consumer_task, return_exceptions=True)
    await r.delete(KEY_PREFIX + "worker:alive")
    await r.aclose()
    print("Воркер остановлен, heartbeat снят.")
